In [ ]:
# Colab setup
'''
********* The best way of colab practice with less cost is that ,
First  mount your Google Drive  to ---> /content/drive   :)
 if you want to load many data for training , you would better to copy files from Google Drive
 In this way you have less latency because of public network latency and data transfer.
 

'''
import tensorflow as tf
import keras

from google.colab import drive
drive.mount("/content/drive")


print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# Cell 3
PROJECT = "/content/drive/MyDrive/market_ml"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
!ls -ltrh "$PROJECT"
!ls -ltrh  "$PROJECT/config"
!ls -ltrh "$PROJECT/src"

In [ ]:
!ls "$PROJECT/data/parquet/model_matrix/sequence_length=120/horizon=15m/scope=continuous/stride=1/sessions=premarket+regular+aftermarket/feature_set=core_v1"

In [ ]:
!python "$PROJECT/src/train_model.py" \
  --config "$PROJECT/config/pipeline.yaml" \
  --show-config

SMOke Test

In [ ]:
!python "$PROJECT/src/train_model.py" \
  --model-matrix-root "$PROJECT/data/parquet/model_matrix" \
  --output-root "$PROJECT/data/models" \
  --symbol nvda \
  --config "$PROJECT/config/pipeline.yaml" \
  --reports "$PROJECT/reports/training" \
  --smoke-test \
  --run-id nvda_core_v1_tf_cpu_smoke

Base line run


In [ ]:
!python "$PROJECT/src/train_model.py" \
  --model-matrix-root "$PROJECT/data/parquet/model_matrix" \
  --output-root "$PROJECT/data/models" \
  --symbol nvda \
  --config "$PROJECT/config/pipeline.yaml" \
  --reports "$PROJECT/reports/training" \
  --run-id nvda_core_v1_tf_gpu_baseline_v1

# change pipeline to have binary class and ignore neutral

In [ ]:
!sed -i 's/label_mode: canonical_3class/label_mode: binary_threshold/' \
  "$PROJECT/config/pipeline.yaml"


In [ ]:
!python "$PROJECT/src/train_model.py" \
  --config "$PROJECT/config/pipeline.yaml" \
  --show-config

## **Change only the label policy**

In [ ]:
!python "$PROJECT/src/train_model.py" \
  --model-matrix-root "$PROJECT/data/parquet/model_matrix" \
  --output-root "$PROJECT/data/models" \
  --symbol nvda \
  --config "$PROJECT/config/pipeline.yaml" \
  --reports "$PROJECT/reports/training" \
  --run-id nvda_core_v1_tf_gpu_binary0_v1

##  After label investigaton and updating files from local driver

In [ ]:
# RUN ON: COLAB
# GPU USAGE: NEGLIGIBLE — verification only
# PARQUET IMPACT: READ-ONLY
# CHANGED PATHS: NONE

PROJECT = "/content/drive/MyDrive/market_ml"

!grep -n 'TRAIN_MODEL_VERSION' "$PROJECT/src/train_model.py" | head

# RUN ON: COLAB
# GPU USAGE: NONE FOR THIS COMMAND
# PARQUET IMPACT: NONE
# CHANGED PATHS: NONE

!python "$PROJECT/src/train_model.py" \
  --config "$PROJECT/config/pipeline.yaml" \
  --show-config
# RUN ON: COLAB
# GPU USAGE: VERIFICATION ONLY
# PARQUET IMPACT: NONE
# CHANGED PATHS: NONE

import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# Smoke test before full run

In [ ]:
# RUN ON: COLAB GPU
# PARQUET IMPACT: READ-ONLY
#
# NEW OUTPUT:
#   data/models/.../run_id=nvda_core_v1_atr075_gpu_smoke/
#   reports/training/...
#
# PURPOSE:
#   Final 1-epoch integration smoke before full baseline.

!python "$PROJECT/src/train_model.py" \
  --model-matrix-root "model_matrix" \
  --label-policy-root "label_policy" \
  --output-root "$PROJECT/data/models" \
  --symbol nvda \
  --config "$PROJECT/config/pipeline.yaml" \
  --reports "$PROJECT/reports/training" \
  --smoke-test \
  --run-id nvda_core_v1_atr075_gpu_smoke

 # This is now a clean controlled comparison with the earlier models. Everything stays fixed except the label definition:

 ## A — canonical 3-class
      balanced accuracy   34.74%
      macro F1            33.97%
      neutral recall       0%

  ## B — binary sign
      balanced accuracy   51.96%
      macro F1            50.61%

  ## C — ATR14 × 0.75 3-class
      ← running now

In [ ]:
# RUN ON: COLAB GPU
# WORKING PROJECT:
#   /content/drive/MyDrive/market_ml
#
# PARQUET IMPACT: READ-ONLY
# CHANGED PARQUET PATHS: NONE
#
# NEW OUTPUTS:
#   data/models/.../run_id=nvda_core_v1_atr075_gpu_baseline_v1/
#   reports/training/...
#
# EXPERIMENT:
#   NVDA
#   core_v1 — 23 features
#   L120 / H15
#   LSTM64 x 2
#   ATR14-relative 3-class labels
#   multiplier = 0.75
#   full TRAIN / VALIDATION
#   TEST sealed

!python "$PROJECT/src/train_model.py" \
  --model-matrix-root "$PROJECT/data/parquet/model_matrix" \
  --label-policy-root "$PROJECT/data/parquet/label_policy" \
  --output-root "$PROJECT/data/models" \
  --symbol nvda \
  --config "$PROJECT/config/pipeline.yaml" \
  --reports "$PROJECT/reports/training" \
  --run-id nvda_core_v1_atr075_gpu_baseline_v1

## Confusion matrix after full run adding ATR for 3class problem

In [ ]:
!find "$PROJECT/reports/training/nvda" \
  -type f \
  -name "*nvda_core_v1_atr075_gpu_baseline_v1*.json" \
  -print
  # RUN ON: COLAB
# GPU: NOT NEEDED
# PARQUET IMPACT: NONE
# CHANGED PATHS: NONE

!cat "$PROJECT/reports/training/nvda/"*nvda_core_v1_atr075_gpu_baseline_v1*.json

# session diagnostic

In [3]:
!python "$PROJECT/src/session_diagnostics_fast.py" \
  --project-root "$PROJECT" \
  --model-matrix-root "$PROJECT/data/parquet/model_matrix" \
  --label-policy-root "$PROJECT/data/parquet/label_policy" \
  --model "$PROJECT/data/models/sequence_length=120/horizon=15m/scope=continuous/stride=1/sessions=premarket+regular+aftermarket/feature_set=core_v1/model=lstm/symbol=nvda/run_id=nvda_core_v1_atr075_gpu_baseline_v1/best_model.keras" \
  --symbol nvda \
  --config "$PROJECT/config/pipeline.yaml" \
  --reports "$PROJECT/reports/session_diagnostics"